In [1]:
# The Iterator Protocol
class CountUp:
  """Count from start to stop"""
  def __init__(self, start, stop):
    self.current = start
    self.stop = stop

  def __iter__(self):
    return self

  def __next__(self):
    if self.current >= self.stop:
      raise StopIteration

    value = self.current
    self.current += 1
    return value

counter = CountUp(1, 5)
for num in counter:
  print(num)

1
2
3
4


**Generators: Easy Iterators with yield**

In [3]:
""" A function that uses yield instead of return. It automatically implements the
iterator protocol"""
def Count_Up(start, stop):
  """Generator version of CountUp"""
  current = start
  while current < stop:
    yield current
    current += 1

for num in Count_Up(1,5):
  print(num)

gen = Count_Up(1,5)
print(next(gen))

1
2
3
4
1


In [8]:
# Memory Efficiency
import sys

def get_numbers_list(n):
  return [i for i in range(n)]

# Method 2
def get_numbers_generator(n):
    for i in range(n):
        yield i
# Comparing memory usage
n = 1_000_000
numbers_list = get_numbers_list(n)
numbers_gen = get_numbers_generator(n)

print(f"List size: {sys.getsizeof(numbers_list)} bytes")
print(f"Generator size: {sys.getsizeof(numbers_gen)} bytes")

# Both can be iterated the same way
for num in numbers_gen:
    if num > 10:
        break


List size: 8448728 bytes
Generator size: 200 bytes


In [ ]:
def read_large_file_bad(filepath):
    """❌ Bad: Loads entire file into memory."""
    with open(filepath, 'r') as f:
        return f.readlines()  # Returns list of ALL lines

def read_large_file_good(filepath):
    """✅ Good: Reads line by line."""
    with open(filepath, 'r') as f:
        for line in f:  # File object is already an iterator!
            yield line.strip()

# Process a 10 GB log file
for line in read_large_file_good('/var/log/huge.log'):
    if 'ERROR' in line:
        print(line)
    # Only one line in memory at a time!

In [10]:
# Generators Expressions
# List comprehension (creates full list)
squares_list = [x**2 for x in range(1000000)]

# Generator expression (computes on-demand)
squares_gen = (x**2 for x in range(1000000))  # Note: parentheses, not brackets

# Both can be used the same way
print(sum(squares_list))
print(sum(squares_gen))

# But generator is more memory-efficien

333332833333500000
333332833333500000


In [ ]:
# PipeLine Processing
def read_log_file(filepath):
    """Read log file line by line."""
    with open(filepath, 'r') as f:
        for line in f:
            yield line.strip()

def parse_log_line(lines):
  """Parse each log line into a dict"""
  for line in lines:
    parts = line.split('|')
    if len(parts) >= 3:
      yield {
          'timestamp': parts[0],
          'level': parts[1],
           'message': parts[2]
      }
def filter_errors(logs):
  for log in logs:
    if log['level'] == 'Error':
      yield log

# Chain generators (pipeline)
log_lines = read_log_file('/var/log/app.log')
parsed_logs = parse_log_line(log_lines)
error_logs = filter_errors(parsed_logs)

# Process errors one at a time (memory-efficient)
for error in error_logs:
    print(error)
    # Only one log entry in memory at a time,
    # even if the file is 100 GB!

In [11]:
# yield from
def flatten(nested_list):
  for item in nested_list:
    if isinstance(item, list):
      yield from flatten(item)
    else:
      yield item

nested = [1, [2, 3, [4, 5]], 6, [7, [8, 9]]]
flat = list(flatten(nested))
print(flat)

[1, 2, 3, 4, 5, 6, 7, 8, 9]


**Real World**

In [12]:
# Pagination
def paginate(items, page_size):
    """Yield pages of items."""
    items = list(items)
    for i in range(0, len(items), page_size):
        yield items[i:i + page_size]

# Usage
all_users = list(range(1, 101))  # 100 users

for page_num, page in enumerate(paginate(all_users, 10), start=1):
    print(f"Page {page_num}: {page}")

Page 1: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Page 2: [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Page 3: [21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
Page 4: [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
Page 5: [41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
Page 6: [51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
Page 7: [61, 62, 63, 64, 65, 66, 67, 68, 69, 70]
Page 8: [71, 72, 73, 74, 75, 76, 77, 78, 79, 80]
Page 9: [81, 82, 83, 84, 85, 86, 87, 88, 89, 90]
Page 10: [91, 92, 93, 94, 95, 96, 97, 98, 99, 100]


In [13]:
import itertools

def secure_paginate(items, page_size):
    # 1. Defend against malicious/bad page sizes
    if not isinstance(page_size, int) or page_size <= 0:
        raise ValueError("Page size must be a positive integer.")

    # 2. Iterate lazily without converting everything to a list at once
    iterator = iter(items)
    while True:
        # Pull only a slice of size 'page_size' from the stream
        page = list(itertools.islice(iterator, page_size))
        if not page:
            break
        yield page


In [14]:
# Batch Processing
def process_in_batches(items, batch_size):
    """Process items in batches."""
    batch = []

    for item in items:
        batch.append(item)

        if len(batch) >= batch_size:
            yield batch
            batch = []

    # Yield remaining items
    if batch:
        yield batch

# Usage: Bulk insert into database
def bulk_insert_users(users):
    """Insert users in batches of 1000."""
    for batch in process_in_batches(users, 1000):
        print(f"Inserting batch of {len(batch)} users")
        # db.bulk_insert(batch)

# Process 10,000 users without loading all into memory
user_generator = (f"user_{i}" for i in range(10000))
bulk_insert_users(user_generator)

Inserting batch of 1000 users
Inserting batch of 1000 users
Inserting batch of 1000 users
Inserting batch of 1000 users
Inserting batch of 1000 users
Inserting batch of 1000 users
Inserting batch of 1000 users
Inserting batch of 1000 users
Inserting batch of 1000 users
Inserting batch of 1000 users


In [15]:
# infinite Sequences
def fibonacci():
    """Generate infinite Fibonacci sequence."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Get first 10 Fibonacci numbers
fib = fibonacci()
first_10 = [next(fib) for _ in range(10)]
print(first_10)  # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

# Or use with itertools
from itertools import islice
first_20 = list(islice(fibonacci(), 20))
print(first_20)

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
